# 消息与提示词模板

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import (
HumanMessage, # 用户消息
SystemMessage, # 系统消息
AIMessage, # AI消息
)

#  读取 .env 配置文件中的信息
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = os.getenv('DEEPSEEK_BASE_URL')
DEEPSEEK_MODEL = os.getenv('DEEPSEEK_MODEL')

# 获取大模型
model = init_chat_model(
    model=DEEPSEEK_MODEL,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)
messages = [
   SystemMessage(content='你是一个信息抽取器。你会收到多条来自不同发言者的 user 消息。每条消息可能带有 name 字段。你的任务是：严格根据每条消息的 name 提取发言者及其观点，并输出JSON。禁止使用“第一个人/第二个人”这种相对称呼。若某条消息没有 name，则输出 unknown。输出格式：{\"speakers\":[{\"name\":\"...\",\"claim\":\"...\"}]}'),
    HumanMessage(content='我认为 1+1=2',name='Bob'),
    HumanMessage(content='我认为 1+1>2',name='Tom'),
    HumanMessage(content='请列出谁说了什么，不要判断对错。',name='audience')
]
response = model.invoke(messages)
response = model.invoke('我叫大白')
response = model.invoke('我叫什么')
print(response.content)

## 对话历史优化

In [76]:
def keep_recent_messages(messages_, max_pairs=3):
    """
    保留最近的 N 轮对话
    :param messages_:
    :param max_pairs: 保留的对话轮数（每轮 = user + assistant）
    :return:
    """
    # 分离 system 和对话
    system_msgs = [m for m in messages_ if m.type == 'system']
    conversation_msgs = [m for m in messages_ if m.type != 'system']

    # 只保留最近的
    recent_msgs = conversation_msgs[-(max_pairs * 2):]

    # 返回 system + 最近对话
    return system_msgs + recent_msgs



long_conversation = [
    SystemMessage(content='你是 Python 导师')
]
# 第一轮
long_conversation.append(HumanMessage(content='什么是列表？用一句解释'))
r1 = model.invoke(long_conversation)
long_conversation.append(AIMessage(content=r1.content))

# # 第二轮
long_conversation.append(HumanMessage(content='列表和元组有什么区别？用一句解释'))
r2 = model.invoke(long_conversation)
long_conversation.append(AIMessage(content=r2.content))

# # 第三轮
long_conversation.append(HumanMessage(content='什么是字典呢？用一句解释'))
r3 = model.invoke(long_conversation)
long_conversation.append(AIMessage(content=r3.content))

print(f'原始消息数：{len(long_conversation)}')

# 只保留最近 2 轮
optimized = keep_recent_messages(long_conversation,max_pairs=2)
print(f'优化后消息数：{len(optimized)}')
print(f'保留的内容：system + 最近2轮对话')

# 添加新的用户问题
optimized.append(HumanMessage(content='我第一个问题问的是什么？'))
# 使用优化后的历史
response = model.invoke(optimized)
print(f'\nAI 回复: {response.content}')

原始消息数：7
优化后消息数：5
保留的内容：system + 最近2轮对话

AI 回复: 你第一个问的是：**“列表和元组有什么区别？用一句解释”**
